In [1]:

# 1. Импорт библиотек

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold



# 2. Загрузка данных

sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)



# 3. Очистка данных

df = df.drop(columns=['Unnamed: 0'])
df = df.fillna(df.median(numeric_only=True))

selector = VarianceThreshold(threshold=0.01)
X_temp = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

selector.fit(X_temp)

low_var_cols = X_temp.columns[~selector.get_support()]
df = df.drop(columns=low_var_cols)



# 4. Проверка формулы SI

df['SI_calc'] = df['CC50, mM'] / df['IC50, mM']
df['SI_diff'] = df['SI'] - df['SI_calc']

print('Максимальное отклонение:', df['SI_diff'].abs().max())



# 5. Подготовка данных

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'SI_calc', 'SI_diff'])
y = df['SI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# 6. Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2:", r2_score(y_test, y_pred_lr))



# 7. Decision Tree

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("\nDecision Tree")
print("MAE:", mean_absolute_error(y_test, y_pred_tree))
print("MSE:", mean_squared_error(y_test, y_pred_tree))
print("R2:", r2_score(y_test, y_pred_tree))



# 8. Random Forest

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("R2:", r2_score(y_test, y_pred_rf))

/tmp/ipykernel_4099/4087079349.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['SI_calc'] = df['CC50, mM'] / df['IC50, mM']
/tmp/ipykernel_4099/4087079349.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['SI_diff'] = df['SI'] - df['SI_calc']


Максимальное отклонение: 1.8629798432812095e-06
Linear Regression
MAE: 224.1295995171399
MSE: 1889660.1824817646
R2: 0.059252627669762026

Decision Tree
MAE: 199.80228386383163
MSE: 1850225.6478798583
R2: 0.07888469440309775

Random Forest
MAE: 191.68646041501373
MSE: 1835018.563436686
R2: 0.08645538084891657


### Что получилось

Максимальное отклонение при проверке формулы:
- **1.86e-06** - SI действительно считается как `CC50 / IC50`

**Linear Regression**
- MAE: **224.1**
- MSE: **1889660.2**
- R2: **0.059**

**Decision Tree**
- MAE: **199.8**
- MSE: **1850225.6**
- R2: **0.079**

**Random Forest**
- MAE: **191.7**
- MSE: **1835018.6**
- R2: **0.086**

### Вывод

- Все модели дают очень слабый результат  
- Лучший R2 всего **0.086**, то есть признаки почти не объясняют SI  
- Random Forest немного лучше остальных, но разница небольшая  

 напрямую предсказывать **SI** по имеющимся дескрипторам практически не получается  

Следующее:
- фиксирую вывод по SI и перехожу к файлам классификации

### Финальный вывод по SI

- Все модели показывают очень низкое качество  
- Лучшая модель: **Random Forest (R2 ≈ 0.09)**  
- Признаки почти не объясняют SI  

Причина:
- **SI — производный показатель (CC50 / IC50)**  
- информация о нём теряется при удалении этих признаков  

 регрессию SI нельзя считать надёжной  

Следующее:
- создаю файл **classification_ic50.ipynb**